In [1]:
from pathlib import Path

import pandas as pd

ORIGINAL_TRAIN_PATH = Path("../../data/out/splits/random/mmlu/train_original.parquet")
DISTILL_PATH = Path("../../data/out/distillation/mmlu_corrected_answer_deepseek_v4_pro_and_others.parquet")
OUT_DIR = Path("../../data/out/splits/random/mmlu/")
OUT_NAME = "train_corrected_answer_deepseek_v4_pro_and_others.parquet"

In [2]:
train_ids = set(pd.read_parquet(ORIGINAL_TRAIN_PATH)["question_id"])
distill_df = pd.read_parquet(DISTILL_PATH)
print(f"Distill rows: {len(distill_df)}, original train ids: {len(train_ids)}")

train_df = distill_df[distill_df["question_id"].isin(train_ids)].reset_index(drop=True)
assert len(train_df) == len(train_ids), f"Expected {len(train_ids)} rows, got {len(train_df)}"
print(f"Filtered train rows: {len(train_df)}")

MAX_REASONING_CHARS = 8192
truncated = (train_df["distill_reasoning"].str.len() > MAX_REASONING_CHARS).sum()
train_df["distill_reasoning"] = train_df["distill_reasoning"].str.slice(0, MAX_REASONING_CHARS)
print(f"Truncated distill_reasoning in {truncated} rows to {MAX_REASONING_CHARS} chars")

Distill rows: 12032, original train ids: 9626
Filtered train rows: 9626
Truncated distill_reasoning in 1921 rows to 8192 chars


In [3]:
KEEP_ONLY_CORRECT = False

if KEEP_ONLY_CORRECT:
    before = len(train_df)
    train_df = train_df[train_df["distill_ans_correct"]].reset_index(drop=True)
    print(f"Kept only correct: {before} -> {len(train_df)} rows")
else:
    print(f"Keeping all rows ({len(train_df)})")

Keeping all rows (9626)


In [4]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(str(OUT_DIR / OUT_NAME), index=False)

print(f"Saved to {OUT_DIR.resolve()}")

Saved to /Users/aigoncharov/dev/sktech/recursive_caft/data/out/splits/random/mmlu
